In [6]:
import os
import pandas as pd
from openpyxl import load_workbook
import re
from datetime import datetime
from tqdm import tqdm

def process_stoxx_files(directory_path):
    """
    读取并处理stoxx europe 600目录下的所有xlsx文件
    
    Parameters:
    directory_path: 文件目录路径
    
    Returns:
    pandas.DataFrame: 合并后的数据框
    """
    
    # 获取所有xlsx文件，排除临时文件（以~$开头的文件）
    path = directory_path
    all_files = [
        os.path.join(path, f) 
        for f in os.listdir(path) 
        if f.endswith('.xlsx') and not f.startswith('~$')
    ]
    
    if not all_files:
        print(f"在目录 {path} 中没有找到xlsx文件")
        return pd.DataFrame()
    
    print(f"找到 {len(all_files)} 个xlsx文件（已排除临时文件）\n")
    
    # 存储所有处理后的数据框
    dfs = []
    
    # 添加进度条
    for file_path in tqdm(all_files, desc="📊 处理Excel文件", unit="文件", ncols=100):
        try:
            # 使用tqdm.write代替print，保持进度条整洁
            # tqdm.write(f"📄 正在处理: {os.path.basename(file_path)}")
            
            # 步骤1: 使用openpyxl读取D2单元格获取Rebalance Period
            wb = load_workbook(filename=file_path, data_only=True)
            ws = wb.active
            d2_value = ws.cell(row=2, column=4).value  # D2单元格
            wb.close()
            
            # 从D2中提取日期
            rebalance_date = None
            if d2_value:
                match = re.search(r'Rebalance Period:\s*(\d{1,2}/\d{1,2}/\d{4})', str(d2_value))
                if match:
                    date_str = match.group(1)
                    # 将MM/DD/YYYY格式转换为YYYY-MM-DD格式
                    rebalance_date = pd.to_datetime(date_str, format='%m/%d/%Y').strftime('%Y-%m-%d')
                else:
                    tqdm.write(f"  ⚠️  警告: 无法从D2单元格提取日期: {d2_value}")
            
            # 步骤2: 使用pandas读取数据表，明确指定engine='openpyxl'
            df = pd.read_excel(file_path, skiprows=3, header=None, engine='openpyxl')
            
            # 检查实际列数
            actual_cols = df.shape[1]
            # tqdm.write(f"  📊 检测到 {actual_cols} 列")
            
            # 根据实际列数动态设置列名
            if actual_cols == 8:
                # 如果是8列，说明没有"Index"列或者某列被合并了
                df.columns = ['Ticker_Part1', 'Ticker_Part2', 'Short Name', 
                             'Market Cap', 'Weight', 'Return', 'Previous In/Out', 'Next In/Out']
            elif actual_cols == 9:
                # 如果是9列，使用原始的列名
                df.columns = ['Ticker_Part1', 'Ticker_Part2', 'Index', 'Short Name', 
                             'Market Cap', 'Weight', 'Return', 'Previous In/Out', 'Next In/Out']
            elif actual_cols >= 8:
                # 如果列数更多，只取前8列
                df = df.iloc[:, :8]
                df.columns = ['Ticker_Part1', 'Ticker_Part2', 'Short Name', 
                             'Market Cap', 'Weight', 'Return', 'Previous In/Out', 'Next In/Out']
            else:
                tqdm.write(f"  ⚠️  警告: 列数不足（{actual_cols}列），跳过该文件")
                continue
            
            # 创建完整的Ticker列（A列 + 空格 + B列）
            # 处理Ticker_Part1和Ticker_Part2，将NaN替换为"NA"
            def format_ticker_part(val):
                """将NaN转换为'NA'，其他值转换为字符串"""
                if pd.isna(val):
                    return 'NA'
                return str(val).strip()
            
            ticker_part1 = df['Ticker_Part1'].apply(format_ticker_part)
            ticker_part2 = df['Ticker_Part2'].apply(format_ticker_part)
            
            df['Ticker'] = ticker_part1 + ' ' + ticker_part2
            
            # 处理Market Cap（去除逗号，转换为数值）
            df['Market Cap'] = pd.to_numeric(
                df['Market Cap'].astype(str).str.replace(',', ''), 
                errors='coerce'
            )
            
            # 处理Weight（去除百分号，转换为小数）
            def convert_weight(val):
                if pd.isna(val):
                    return None
                val_str = str(val).strip().replace('%', '')
                try:
                    return float(val_str) / 100
                except:
                    return None
            
            df['Weight'] = df['Weight'].apply(convert_weight)
            
            # 处理Monthly Return（处理括号表示负数，去除百分号）
            def convert_return(val):
                if pd.isna(val):
                    return None
                val_str = str(val).strip().replace('%', '')
                # 处理括号表示负数的情况
                if val_str.startswith('(') and val_str.endswith(')'):
                    val_str = '-' + val_str[1:-1]
                try:
                    return float(val_str) / 100
                except:
                    return None
            
            df['Monthly Return'] = df['Return'].apply(convert_return)
            
            # 添加Rebalance Period列
            df['Rebalance Period'] = rebalance_date
            
            # 重命名Short Name为Name
            df['Name'] = df['Short Name']
            
            # 选择最终需要的列
            final_df = df[['Rebalance Period', 'Ticker', 'Name', 'Market Cap', 'Weight', 'Monthly Return']]
            
            # 去除可能的空行
            final_df = final_df.dropna(subset=['Ticker', 'Name'], how='all')
            
            dfs.append(final_df)
            # tqdm.write(f"  ✅ 成功处理 {len(final_df)} 行数据\n")
            
        except Exception as e:
            tqdm.write(f"  ❌ 错误: 处理文件 {os.path.basename(file_path)} 时出错: {str(e)}\n")
            continue
    
    # 合并所有数据框
    print("\n" + "="*80)
    if dfs:
        result_df = pd.concat(dfs, ignore_index=True)
        print(f"✅ 总共合并了 {len(result_df):,} 行数据，来自 {len(dfs)} 个文件")
        return result_df
    else:
        print("❌ 没有成功处理任何文件")
        return pd.DataFrame()

# 执行处理
directory = r'C:\GoogleDrive\TP\screen\bench\sp_500'
result_df = process_stoxx_files(directory)
result_df.drop_duplicates(subset=['Rebalance Period', 'Ticker'], keep='first')

# 显示结果摘要
if not result_df.empty:
    print("="*80)
    print("\n📊 数据框摘要:")
    print(f"   总行数: {len(result_df):,}")
    print(f"   总列数: {len(result_df.columns)}")
    print(f"   列名: {list(result_df.columns)}")
    
    unique_periods = sorted(result_df['Rebalance Period'].unique())
    print(f"\n📅 唯一的Rebalance Period日期 (共{len(unique_periods)}个):")
    if len(unique_periods) > 0:
        print(f"   最早: {unique_periods[0]}")
        print(f"   最晚: {unique_periods[-1]}")
        if len(unique_periods) <= 10:
            print(f"   所有日期: {unique_periods}")
    
    print("\n📈 前10行数据:")
    print(result_df.head(10).to_string())
    
    # 显示包含"NA"的Ticker示例
    na_tickers = result_df[result_df['Ticker'].str.contains(' NA', na=False)]
    if not na_tickers.empty:
        print(f"\n🔍 包含 'NA' 的Ticker示例（共{len(na_tickers)}条）:")
        print(na_tickers.head(5)[['Ticker', 'Name']].to_string())
    
    print("\n🔍 数据类型:")
    print(result_df.dtypes)
    
    print("\n📉 数值列统计:")
    print(result_df[['Market Cap', 'Weight', 'Monthly Return']].describe())
    
    # 检查缺失值
    print("\n🔎 缺失值统计:")
    print(result_df.isnull().sum())
    
    # 可选：保存为PICKLE文件
    output_path = r'C:\GoogleDrive\TP\screen\bench\sp_500.pickle'
    result_df.to_pickle(output_path)
    print(f"\n💾 数据已保存到: {output_path}")
    print("="*80)
else:
    print("\n⚠️  建议检查:")
    print("   1. 确认Excel文件格式是否正确")
    print("   2. 打开一个文件查看实际的列结构")

result_df.drop_duplicates(subset="Ticker")[["Ticker", "Name"]].to_excel(r"C:\GoogleDrive\TP\screen\bench\sp_500.xlsx", index=False)

找到 91 个xlsx文件（已排除临时文件）



📊 处理Excel文件: 100%|███████████████████████████████████████████| 91/91 [00:11<00:00,  7.75文件/s]
C:\Users\jingx\AppData\Local\Temp\ipykernel_37656\4025586710.py:152: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result_df = pd.concat(dfs, ignore_index=True)



✅ 总共合并了 45,412 行数据，来自 91 个文件

📊 数据框摘要:
   总行数: 45,412
   总列数: 6
   列名: ['Rebalance Period', 'Ticker', 'Name', 'Market Cap', 'Weight', 'Monthly Return']

📅 唯一的Rebalance Period日期 (共90个):
   最早: 2010-01-31
   最晚: 2017-06-30

📈 前10行数据:
  Rebalance Period       Ticker             Name    Market Cap    Weight  Monthly Return
0       2017-06-30  1697067D US  DOW CHEMICAL CO  6.751366e+10  0.003380             NaN
1       2017-06-30  1715651D US         EIDP INC  6.131486e+10  0.003070             NaN
2       2017-06-30  1831877D US     XL GROUP LTD  1.003349e+10  0.000502             NaN
3       2017-06-30  1856613D US      MONSANTO CO  4.556089e+10  0.002281             NaN
4       2017-06-30  1922150D US     LINDE INC/CT  3.314109e+10  0.001659             NaN
5       2017-06-30  1927294D US  L3 TECHNOLOGIES  1.140089e+10  0.000571             NaN
6       2017-06-30  2078185D US     TIFFANY & CO  1.025224e+10  0.000513             NaN
7       2017-06-30  2326248D US           CA INC  1.272

In [1]:
import pandas as pd
bench_sp500 = pd.read_pickle(r'C:\GoogleDrive\TP\screen\bench\sp_500.pickle')
bench_stoxx = pd.read_pickle(r'C:\GoogleDrive\TP\screen\bench\stoxx_europe_600.pickle')

In [3]:
mapping_stoxx = pd.read_excel(r"C:\GoogleDrive\TP\screen\bench\stoxx_europe_600_tickers_names.xlsx")[['Ticker', 'ISIN']]
mapping_sp500 = pd.read_excel(r"C:\GoogleDrive\TP\screen\bench\sp500_tickers_names.xlsx")[['Ticker', 'ISIN']]
mapping_sup = pd.read_excel(r"C:\GoogleDrive\TP\screen\bench\W_14_Basic.xlsx")[['Ticker', 'ISIN']]

In [4]:
mapping = pd.concat([mapping_sup, mapping_sp500, mapping_stoxx]).drop_duplicates(subset='Ticker', keep='first')


In [5]:
mapping = mapping[['Ticker', 'ISIN']]

In [6]:
bench_sp500 = bench_sp500.merge(mapping, on='Ticker', how='left')
bench_sp500.rename(columns={'Weight': 'Weight in SP500'}, inplace=True)    
bench_stoxx = bench_stoxx.merge(mapping, on='Ticker', how='left')
bench_stoxx.rename(columns={'Weight': 'Weight in STOXX EUROPE 600'}, inplace=True)

In [7]:
bench_stoxx = bench_stoxx[['Rebalance Period', 'ISIN', 'Weight in STOXX EUROPE 600']]
bench_sp500 = bench_sp500[['Rebalance Period', 'ISIN', 'Weight in SP500']]
bench = pd.concat([bench_sp500, bench_stoxx], axis=0)
bench.rename(columns={'Rebalance Period' : 'Date'}, inplace=True)
bench['Date'] = pd.to_datetime(bench['Date'])

In [9]:
bench.to_pickle(r'C:\GoogleDrive\TP\screen\bench\bench_sp500_stoxx.pickle')

In [23]:
import pandas as pd
bench = pd.read_pickle(r'C:\GoogleDrive\TP\screen\bench\bench_sp500_stoxx.pickle')
screen = pd.read_parquet(r"C:\GoogleDrive\TP\screen\screen_aggregate_new.parquet")

In [24]:
bench['Date'] = bench['Date'] + pd.offsets.MonthBegin(1)              # 先转换为第二个月的第一天
bench['Date'] = bench['Date'] + pd.offsets.MonthEnd(-1)   

In [25]:
screen[['Weight in SP500', 'Weight in STOXX EUROPE 600']] = screen[['Weight in SP500', 'Weight in STOXX EUROPE 600']].replace({0 : pd.NA})

In [26]:
screen.reset_index(inplace=True)
screen.drop_duplicates(subset=['ISIN', 'Date'], inplace=True)
bench.drop_duplicates(subset=['ISIN', 'Date'], inplace=True)

In [27]:
screen.set_index(['ISIN', 'Date'], inplace=True)
bench.set_index(['ISIN', 'Date'], inplace=True)

In [28]:
screen[['Weight in SP500', 'Weight in STOXX EUROPE 600']] = screen[['Weight in SP500', 'Weight in STOXX EUROPE 600']].combine_first(bench[['Weight in SP500', 'Weight in STOXX EUROPE 600']])
screen[['Weight in SP500', 'Weight in STOXX EUROPE 600']].update(bench[['Weight in SP500', 'Weight in STOXX EUROPE 600']])

In [29]:
bench.reset_index(inplace=True)
screen.reset_index(inplace=True)
screen.set_index(['ISIN'], inplace=True)

In [ ]:
screen[screen['Date'] == ]

In [30]:
screen.dropna(subset="Weight in STOXX EUROPE 600")['Date'].unique()

<DatetimeArray>
['2009-06-30 00:00:00', '2009-09-30 00:00:00', '2009-12-31 00:00:00',
 '2010-01-31 00:00:00', '2010-02-28 00:00:00', '2010-03-31 00:00:00',
 '2010-04-30 00:00:00', '2010-05-31 00:00:00', '2010-06-30 00:00:00',
 '2010-07-31 00:00:00',
 ...
 '2025-02-28 00:00:00', '2025-03-31 00:00:00', '2025-04-30 00:00:00',
 '2025-05-31 00:00:00', '2025-06-30 00:00:00', '2025-07-31 00:00:00',
 '2025-08-31 00:00:00', '2025-09-30 00:00:00', '2025-10-31 00:00:00',
 '2025-11-30 00:00:00']
Length: 194, dtype: datetime64[ns]

In [31]:
screen.to_parquet(r"C:\GoogleDrive\TP\screen\screen_aggregate_new.parquet")

In [2]:
import pandas as pd
screen = pd.read_parquet(r"C:\GoogleDrive\TP\screen\screen_aggregate_new.parquet")
screen = screen[screen['Date'] >= '2010-01-01']
screen.to_parquet(r"C:\GoogleDrive\TP\ML_Enhanced\Input_files\screen_aggregate.parquet")